# 02. Data Preprocessing & Encoding Pipeline

## Mục tiêu
Thực hiện tiền xử lý dữ liệu theo đúng quy chuẩn đã chốt trong `AGENT_SPEC.md`:
1. **Stratified Train/Test Split**: Giữ nguyên tỷ lệ fraud giữa tập train và test.
2. **Feature Engineering & Cleaning**: Loại bỏ các định danh cá nhân / ID giả lập, trích xuất đặc trưng thời gian và độ tuổi.
3. **Encoding Pipeline (spec mục 3)**:
   - **One-hot Encoding**: Áp dụng cho biến low-cardinality (`gender`, `category`, `state`).
   - **Stratified K-fold Target Encoding**: Áp dụng cho biến high-cardinality (`merchant`, `city`, `job`) với smoothing, **bắt buộc dùng `StratifiedKFold`**.
   - **Thứ tự bắt buộc**: Fit chỉ trên train (out-of-fold), test transform dùng full mapping từ toàn bộ train.
4. **Lưu dữ liệu**: Xuất các tập đã tiền xử lý vào `data/processed/`.

In [1]:
# 1. Setup & Imports
import sys
from pathlib import Path

# Đảm bảo import được module từ src/
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

from src.config import (
    RAW_DATA_DIR, PROCESSED_DATA_DIR, TRAIN_FILE, TEST_FILE,
    SEED, TARGET_COL, ONEHOT_COLS, TARGET_ENCODE_COLS,
)
from src.data.encoding import encode_train, encode_test

print('Config loaded successfully.')
print(f'SEED: {SEED}')
print(f'Target column: {TARGET_COL}')
print(f'One-hot columns: {ONEHOT_COLS}')
print(f'Target encode columns: {TARGET_ENCODE_COLS}')

Config loaded successfully.
SEED: 42
Target column: is_fraud
One-hot columns: ['gender', 'category', 'state']
Target encode columns: ['merchant', 'city', 'job']


--- 
## 2. Load & Clean Raw Data
Gộp train và test từ raw data Sparkov, sau đó thực hiện **Stratified Random Split** theo quyết định đã chốt.

In [2]:
# 2. Load raw files
print('Đang đọc dữ liệu raw từ data/raw/...')
df_raw_train = pd.read_csv(RAW_DATA_DIR / TRAIN_FILE)
df_raw_test = pd.read_csv(RAW_DATA_DIR / TEST_FILE)

df_all = pd.concat([df_raw_train, df_raw_test], ignore_index=True)
print(f'Tổng số giao dịch: {len(df_all):,} dòng × {df_all.shape[1]} cột')
print(f'Tỷ lệ fraud tổng thể: {df_all[TARGET_COL].mean():.4%}')

Đang đọc dữ liệu raw từ data/raw/...


Tổng số giao dịch: 1,852,394 dòng × 23 cột
Tỷ lệ fraud tổng thể: 0.5210%


In [3]:
# 3. Feature Engineering & Selection
def preprocess_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    # Bỏ các cột ID và thông tin cá nhân giả lập không có tính khái quát hóa.
    # `zip`: 98.6% mã chỉ có đúng 1 thẻ (gần như mã khách hàng, không phải vùng) và mỗi mã ứng đúng
    # 1 cặp (lat, long) + 1 city — thông tin vùng đã có ở lat/long/city/state. Để dạng số thì vô nghĩa
    # với LR (từng là feature SHAP lớn nhất của LR), encode thì thành học thuộc khách hàng.
    drop_cols = ['Unnamed: 0', 'trans_num', 'cc_num', 'first', 'last', 'street', 'zip']
    data = data.drop(columns=[c for c in drop_cols if c in data.columns])

    # Feature Engineering thời gian & tuổi
    trans_year = None
    if 'trans_date_trans_time' in data.columns:
        trans_dt = pd.to_datetime(data['trans_date_trans_time'])
        trans_year = trans_dt.dt.year
        data['hour'] = trans_dt.dt.hour
        data['day_of_week'] = trans_dt.dt.dayofweek
        data.drop(columns=['trans_date_trans_time'], inplace=True)

    if 'dob' in data.columns:
        dob_dt = pd.to_datetime(data['dob'])
        # Tuổi xấp xỉ = năm giao dịch - năm sinh. Trước đây hardcode "2020 - năm sinh": dữ liệu
        # trải 2019-01 đến 2020-12 (~50% dòng thuộc 2019), nên mọi dòng 2019 bị tính THỪA 1 tuổi
        # so với tuổi thật tại thời điểm giao dịch. Phải dùng năm giao dịch THỰC của từng dòng.
        if trans_year is None:
            trans_year = pd.to_datetime(df['trans_date_trans_time']).dt.year
        data['age'] = trans_year - dob_dt.dt.year
        data.drop(columns=['dob'], inplace=True)

    if 'unix_time' in data.columns:
        data.drop(columns=['unix_time'], inplace=True)

    return data

df_clean = preprocess_features(df_all)
print('Features sau khi làm sạch:')
print(df_clean.dtypes)
print(f'Kích thước mới: {df_clean.shape}')

Features sau khi làm sạch:
merchant        object
category        object
amt            float64
gender          object
city            object
state           object
lat            float64
long           float64
city_pop         int64
job             object
merch_lat      float64
merch_long     float64
is_fraud         int64
hour             int32
day_of_week      int32
age              int32
dtype: object
Kích thước mới: (1852394, 16)


--- 
## 3. Stratified Train / Test Split
Chia 80% train / 20% test theo phân tầng target `is_fraud`.

In [4]:
# 4. Stratified Split
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.20,
    stratify=df_clean[TARGET_COL],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train set: {len(train_df):,} dòng, Fraud: {train_df[TARGET_COL].sum():,} ({train_df[TARGET_COL].mean():.4%})')
print(f'Test set:  {len(test_df):,} dòng, Fraud: {test_df[TARGET_COL].sum():,} ({test_df[TARGET_COL].mean():.4%})')

Train set: 1,481,915 dòng, Fraud: 7,721 (0.5210%)
Test set:  370,479 dòng, Fraud: 1,930 (0.5209%)


--- 
## 4. Encoding: One-Hot & Stratified K-Fold Target Encoding
- Fit chỉ trên `train_df` bằng `encode_train()`.
- Transform `test_df` bằng `encode_test()` sử dụng `encoding_maps` từ train.

In [5]:
# 5. Fit & Transform Encoding
print('Đang thực hiện Encoding trên tập train (Stratified K-fold)...')
encoded_train, encoding_maps = encode_train(
    train_df,
    target_col=TARGET_COL,
    onehot_cols=ONEHOT_COLS,
    target_encode_cols=TARGET_ENCODE_COLS,
    n_splits=5,
    smoothing=10,
    random_state=SEED,
)

print('Đang transform tập test với mapping từ train...')
encoded_test = encode_test(
    test_df,
    encoding_maps,
    target_encode_cols=TARGET_ENCODE_COLS,
)

print('Hoàn thành encoding!')
print(f'Shape train: {encoded_train.shape}')
print(f'Shape test:  {encoded_test.shape}')
assert list(encoded_train.columns) == list(encoded_test.columns), 'Cột train và test không khớp!'

Đang thực hiện Encoding trên tập train (Stratified K-fold)...


Đang transform tập test với mapping từ train...
Hoàn thành encoding!
Shape train: (1481915, 80)
Shape test:  (370479, 80)


--- 
## 5. Lưu dữ liệu đã tiền xử lý
Lưu kết quả ra `data/processed/` dưới dạng `.parquet` và `.csv` để phục vụ modeling.

In [6]:
# 6. Save processed datasets
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_out_parquet = PROCESSED_DATA_DIR / 'train_encoded.parquet'
test_out_parquet = PROCESSED_DATA_DIR / 'test_encoded.parquet'
maps_out = PROCESSED_DATA_DIR / 'encoding_maps.joblib'

encoded_train.to_parquet(train_out_parquet, index=False)
encoded_test.to_parquet(test_out_parquet, index=False)
joblib.dump(encoding_maps, maps_out)

print(f'✅ Đã lưu train (encoded): {train_out_parquet}')
print(f'✅ Đã lưu test (encoded):  {test_out_parquet}')
print(f'✅ Đã lưu maps:  {maps_out}')

# Lưu thêm bản RAW (chưa encode, chỉ mới split + feature engineering).
# Bắt buộc cho notebook 04 (CIES) — spec mục 7.1 yêu cầu encoding phải
# fit lại từ đầu trên mỗi bootstrap resample, KHÔNG được tái sử dụng
# encoding đã tính sẵn ở đây.
train_raw_parquet = PROCESSED_DATA_DIR / 'train_raw.parquet'
test_raw_parquet = PROCESSED_DATA_DIR / 'test_raw.parquet'

train_df.to_parquet(train_raw_parquet, index=False)
test_df.to_parquet(test_raw_parquet, index=False)

print(f'✅ Đã lưu train (raw, chưa encode): {train_raw_parquet}')
print(f'✅ Đã lưu test (raw, chưa encode):  {test_raw_parquet}')

✅ Đã lưu train (encoded): /Users/thetrung/Projects/Fraud Detection - CIES/data/processed/train_encoded.parquet
✅ Đã lưu test (encoded):  /Users/thetrung/Projects/Fraud Detection - CIES/data/processed/test_encoded.parquet
✅ Đã lưu maps:  /Users/thetrung/Projects/Fraud Detection - CIES/data/processed/encoding_maps.joblib


✅ Đã lưu train (raw, chưa encode): /Users/thetrung/Projects/Fraud Detection - CIES/data/processed/train_raw.parquet
✅ Đã lưu test (raw, chưa encode):  /Users/thetrung/Projects/Fraud Detection - CIES/data/processed/test_raw.parquet


--- 
## 6. Dataset Phụ (ULB Credit Card Fraud)
Tải và tiền xử lý dataset phụ ULB qua `kagglehub`.

In [7]:
# 7. Tải và xử lý dataset ULB
import kagglehub
from src.config import KAGGLE_DATASET_ULB, ULB_FILE

try:
    print(f'Đang tải dataset {KAGGLE_DATASET_ULB} từ Kaggle...')
    ulb_path = kagglehub.dataset_download(KAGGLE_DATASET_ULB)
    ulb_file = Path(ulb_path) / ULB_FILE
    df_ulb = pd.read_csv(ulb_file)
    print(f'ULB Dataset loaded: {df_ulb.shape[0]:,} dòng × {df_ulb.shape[1]} cột')
    print(f'Tỷ lệ fraud: {df_ulb["Class"].mean():.4%}')
    
    # Stratified split 80/20
    ulb_train, ulb_test = train_test_split(
        df_ulb, test_size=0.20, stratify=df_ulb['Class'], random_state=SEED
    )
    ulb_train.to_parquet(PROCESSED_DATA_DIR / 'ulb_train.parquet', index=False)
    ulb_test.to_parquet(PROCESSED_DATA_DIR / 'ulb_test.parquet', index=False)
    print('✅ Đã lưu processed ULB dataset thành công!')
except Exception as e:
    print(f'Lưu ý về ULB dataset: {e}')
    print('Có thể tải thủ công hoặc chạy trên Kaggle/Colab có kết nối Kaggle API.')

Đang tải dataset mlg-ulb/creditcardfraud từ Kaggle...


ULB Dataset loaded: 284,807 dòng × 31 cột
Tỷ lệ fraud: 0.1727%


✅ Đã lưu processed ULB dataset thành công!
